In [0]:
%sql
CREATE DATABASE IF NOT EXISTS f1_processed;


In [0]:

# COMMAND ----------

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, explode, current_timestamp
from delta.tables import DeltaTable

# COMMAND ----------

# MAGIC %md
# MAGIC ### 1. Define Corrected Ingestion Schema
# MAGIC *API data types for 'lat' and 'long' are text strings. We ingest them as strings first to prevent nullification errors.*

# COMMAND ----------

circuits_schema = StructType(fields=[
    StructField("MRData", StructType([
        StructField("CircuitTable", StructType([
            StructField("Circuits", StringType(), True) # Ingest array block as string first for safe parsing
        ]), True)
    ]), True)
])

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2. Extract Data using Folder-Level Wildcard Ingestion

# COMMAND ----------

# Read all JSON files dropped by ADF inside the circuits folder
raw_df = spark.read \
    .schema(circuits_schema) \
    .json("/mnt/f1-raw/circuits/*")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 3. Parse, Explode, and Flatten Nested Data

# COMMAND ----------

from pyspark.sql.functions import from_json, ArrayType

# Define structural schema of individual objects inside the Circuits array
circuit_element_schema = StructType([
    StructField("circuitId", StringType(), False),
    StructField("circuitName", StringType(), True),
    StructField("Location", StructType([
        StructField("lat", StringType(), True),
        StructField("long", StringType(), True),
        StructField("locality", StringType(), True),
        StructField("country", StringType(), True)
    ]), True)
])

# Unpack the string block into a structured array
parsed_df = raw_df.withColumn(
    "circuit_array", 
    from_json(col("MRData.CircuitTable.Circuits"), ArrayType(circuit_element_schema))
)

# Explode the array into individual row records
exploded_df = parsed_df.select(explode(col("circuit_array")).alias("circuit"))

# Select, rename (snake_case), and cast data types to match analytical requirements
silver_circuits_df = exploded_df.select(
    col("circuit.circuitId").alias("circuit_id"),
    col("circuit.circuitName").alias("name"),
    col("circuit.Location.locality").alias("location"),
    col("circuit.Location.country").alias("country"),
    col("circuit.Location.lat").cast("double").alias("latitude"),   # Safely cast to numerical float
    col("circuit.Location.long").cast("double").alias("longitude"), # Safely cast to numerical float
    current_timestamp().alias("ingestion_date")                     # Operational audit column
).dropDuplicates(["circuit_id"])                                    # Deduplicate rows in-memory

# Display data preview to confirm extraction
display(silver_circuits_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 4. Idempotent Upsert (MERGE) into Silver Delta Table

# COMMAND ----------

# Target table in Hive Metastore
target_table = "hive_metastore.f1_processed.circuits"

# Check if target table is empty/uninitialized
if not spark.catalog.tableExists(target_table):
    # First-time initialization run
    silver_circuits_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    print("Database table successfully initialized and loaded.")
else:
    # Production Merge Routine: Prevents duplication of data from multiple files
    tgt_table = DeltaTable.forName(spark, target_table)

    tgt_table.alias("tgt") \
        .merge(
            source = silver_circuits_df.alias("src"), 
            condition = "tgt.circuit_id = src.circuit_id"
        ) \
        .whenMatchedUpdate(set = {
            "name": "src.name", 
            "location": "src.location", 
            "country": "src.country", 
            "latitude": "src.latitude", 
            "longitude": "src.longitude", 
            "ingestion_date": "src.ingestion_date"
        }) \
        .whenNotMatchedInsert(values = {
            "circuit_id": "src.circuit_id", 
            "name": "src.name", 
            "location": "src.location", 
            "country": "src.country", 
            "latitude": "src.latitude", 
            "longitude": "src.longitude", 
            "ingestion_date": "src.ingestion_date"
        }) \
        .execute()

    print("Incremental deduplicated update executed successfully via Delta Merge.")